# Chapter 36 Walkthrough: Flagship Research Capstone — From Raw Survey CSV to Publication-Ready Paper & Reproducible Artifact

**Book Reference:** Chapter 36 (Grand Finale Capstone), Sections 36.1 – 36.4  
Run every cell top-to-bottom (`Shift + Enter`) alongside the textbook dissections. This grand-finale walkthrough synthesizes Parts I–VII on `mch_survey_sample`: data cleaning, Journal Table 1, multivariable adjusted Odds Ratios (`statsmodels`), a 4-model 5-fold `GroupKFold` benchmark (LR, RF, HGB, PyTorch MLP), out-of-fold Rural vs. Urban fairness auditing, Decision Curve Analysis (Vickers & Elkin, 2006), a <=300-word structured journal abstract, and a SHA-256 verified artifact release strictly inside `tempfile.TemporaryDirectory()`.

In [ ]:
# @title Step 0: Click Play (▶) to Load Capstone Research Stack
!pip install -q sharukh-tutor
import hashlib
import json
import tempfile
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    brier_score_loss,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import sharukh_tutor as tutor

torch.set_num_threads(1)
torch.manual_seed(42)
np.random.seed(42)

mch_raw = tutor.load_data("mch_survey_sample", as_pandas=True)
print("Loaded mch_survey_sample:", mch_raw.shape,
      "| Clusters:", mch_raw["cluster_id"].nunique())


## Section 36.3.1 — Survey Cleaning, Journal Table 1 & Adjusted Odds Ratios

In [ ]:
WEALTH_MAP = {
    "Poorest": 1, "Poorer": 2, "Middle": 3,
    "Richer": 4, "Richest": 5,
}
FEATURES = [
    "maternal_age_years", "anc_visits",
    "maternal_hb_g_dl", "is_urban", "wealth_rank",
]


def clean_mch_survey(mch_df):
    """Clean survey sentinels & casing without mutating input."""
    df = mch_df.copy()
    anc = pd.to_numeric(df["anc_visits"], errors="coerce")
    df["anc_visits"] = anc.replace(999, np.nan)
    df["residence"] = (
        df["residence"].astype(str).str.strip().str.title()
    )
    df["is_urban"] = (df["residence"] == "Urban").astype(int)
    df["wealth_rank"] = df["wealth_quintile"].map(WEALTH_MAP)
    df = df.dropna(subset=["child_stunted"]).copy()
    df["child_stunted"] = df["child_stunted"].astype(int)
    return df


def build_publication_table1(mch_df):
    """Build Journal Table 1 comparing non-stunted vs stunted."""
    df = clean_mch_survey(mch_df)
    g0 = df[df["child_stunted"] == 0]
    g1 = df[df["child_stunted"] == 1]
    rows = []
    for col in ["maternal_age_years", "anc_visits", "maternal_hb_g_dl"]:
        s0, s1 = g0[col].dropna(), g1[col].dropna()
        _, p_val = stats.ttest_ind(s0, s1, equal_var=False)
        rows.append({
            "variable": col,
            "not_stunted": f"{s0.mean():.2f} ± {s0.std(ddof=1):.2f}",
            "stunted": f"{s1.mean():.2f} ± {s1.std(ddof=1):.2f}",
            "p_value": round(float(p_val), 4),
        })
    ct = pd.crosstab(df["child_stunted"], df["is_urban"])
    _, p_chi, _, _ = stats.chi2_contingency(ct, correction=False)
    c0, pct0 = int(g0["is_urban"].sum()), 100.0 * g0["is_urban"].mean()
    c1, pct1 = int(g1["is_urban"].sum()), 100.0 * g1["is_urban"].mean()
    rows.append({
        "variable": "is_urban",
        "not_stunted": f"{c0} ({pct0:.1f}%)",
        "stunted": f"{c1} ({pct1:.1f}%)",
        "p_value": round(float(p_chi), 4),
    })
    return pd.DataFrame(rows)


table1_df = build_publication_table1(mch_raw)
print("=== Journal Table 1 ===")
print(table1_df.to_string(index=False))

df_clean = clean_mch_survey(mch_raw)
cc = df_clean[["child_stunted"] + FEATURES].dropna()
X_sm = sm.add_constant(cc[FEATURES].astype(float))
logit_res = sm.Logit(
    cc["child_stunted"].astype(float), X_sm
).fit(disp=False)
ci = logit_res.conf_int()
aor_rows = []
for feat in FEATURES:
    aor_rows.append({
        "feature": feat,
        "aor": round(float(np.exp(logit_res.params[feat])), 4),
        "ci_low": round(float(np.exp(ci.loc[feat, 0])), 4),
        "ci_high": round(float(np.exp(ci.loc[feat, 1])), 4),
        "p_value": round(float(logit_res.pvalues[feat]), 4),
    })
aor_df = pd.DataFrame(aor_rows)
print(f"\n=== Adjusted Odds Ratios (n={len(cc)} complete cases) ===")
print(aor_df.to_string(index=False))


## Section 36.3.2 — 4-Model Cluster-Grouped Benchmark (`GroupKFold` + PyTorch MLP)

In [ ]:
def _fit_predict_mlp_fold(X_tr_df, y_tr, X_va_df, fold_seed):
    """Train leak-free 16-unit PyTorch MLP on one CV fold."""
    imp = SimpleImputer(strategy="median")
    scl = StandardScaler()
    X_tr_s = scl.fit_transform(imp.fit_transform(X_tr_df))
    X_va_s = scl.transform(imp.transform(X_va_df))
    torch.set_num_threads(1)
    torch.manual_seed(fold_seed)
    net = nn.Sequential(
        nn.Linear(X_tr_s.shape[1], 16),
        nn.ReLU(),
        nn.Linear(16, 1),
    )
    opt = torch.optim.Adam(net.parameters(), lr=0.02)
    loss_fn = nn.BCEWithLogitsLoss()
    xt = torch.tensor(X_tr_s, dtype=torch.float32)
    yt = torch.tensor(y_tr, dtype=torch.float32).unsqueeze(1)
    xv = torch.tensor(X_va_s, dtype=torch.float32)
    net.train()
    for _ in range(30):
        opt.zero_grad()
        loss_fn(net(xt), yt).backward()
        opt.step()
    net.eval()
    with torch.no_grad():
        return torch.sigmoid(net(xv)).squeeze(-1).numpy()


def run_capstone_model_benchmark(mch_df, seed=42):
    """Benchmark 4 model families under 5-fold cluster GroupKFold."""
    df = clean_mch_survey(mch_df)
    X = df[FEATURES].astype(float)
    y = df["child_stunted"].to_numpy()
    groups = df["cluster_id"].to_numpy()
    gkf = GroupKFold(n_splits=5)

    sk_builders = {
        "LogisticRegression": lambda: Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("scl", StandardScaler()),
            ("clf", LogisticRegression(max_iter=1000, random_state=seed)),
        ]),
        "RandomForestClassifier": lambda: Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("clf", RandomForestClassifier(
                n_estimators=80, max_depth=4, random_state=seed
            )),
        ]),
        "HistGradientBoostingClassifier": lambda: Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("clf", HistGradientBoostingClassifier(
                max_iter=50, max_depth=3, random_state=seed
            )),
        ]),
    }
    rows = []
    for name, make_pipe in sk_builders.items():
        aucs, briers = [], []
        for tr, va in gkf.split(X, y, groups):
            pipe = make_pipe()
            pipe.fit(X.iloc[tr], y[tr])
            probs = pipe.predict_proba(X.iloc[va])[:, 1]
            aucs.append(roc_auc_score(y[va], probs))
            briers.append(brier_score_loss(y[va], probs))
        rows.append({
            "model": name,
            "mean_auc": round(float(np.mean(aucs)), 4),
            "std_auc": round(float(np.std(aucs)), 4),
            "mean_brier": round(float(np.mean(briers)), 4),
        })

    mlp_aucs, mlp_briers = [], []
    for fold_i, (tr, va) in enumerate(gkf.split(X, y, groups)):
        probs = _fit_predict_mlp_fold(
            X.iloc[tr], y[tr], X.iloc[va], seed + fold_i
        )
        mlp_aucs.append(roc_auc_score(y[va], probs))
        mlp_briers.append(brier_score_loss(y[va], probs))
    rows.append({
        "model": "PyTorch_MLP",
        "mean_auc": round(float(np.mean(mlp_aucs)), 4),
        "std_auc": round(float(np.std(mlp_aucs)), 4),
        "mean_brier": round(float(np.mean(mlp_briers)), 4),
    })
    return (
        pd.DataFrame(rows)
        .sort_values("mean_auc", ascending=False)
        .reset_index(drop=True)
    )


leaderboard_df = run_capstone_model_benchmark(mch_raw, seed=42)
print(leaderboard_df.to_string(index=False))


## Section 36.3.3 — Out-of-Fold Subgroup Fairness Audit & Results Markdown

In [ ]:
X_all = df_clean[FEATURES].astype(float)
y_all = df_clean["child_stunted"].to_numpy()
groups_all = df_clean["cluster_id"].to_numpy()
gkf = GroupKFold(n_splits=5)

oof_rf = np.zeros(len(y_all), dtype=float)
for tr, va in gkf.split(X_all, y_all, groups_all):
    pipe = Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("clf", RandomForestClassifier(
            n_estimators=80, max_depth=4, random_state=42
        )),
    ])
    pipe.fit(X_all.iloc[tr], y_all[tr])
    oof_rf[va] = pipe.predict_proba(X_all.iloc[va])[:, 1]

fpr, tpr, thresh = roc_curve(y_all, oof_rf)
youden_t = float(thresh[int(np.argmax(tpr - fpr))])
y_hat = (oof_rf >= youden_t).astype(int)
rural_m = df_clean["is_urban"].to_numpy() == 0
urban_m = df_clean["is_urban"].to_numpy() == 1

r_pos = float(y_hat[rural_m].mean())
u_pos = float(y_hat[urban_m].mean())
r_tpr = float(y_hat[rural_m & (y_all == 1)].mean())
u_tpr = float(y_hat[urban_m & (y_all == 1)].mean())
fairness_dict = {
    "threshold": round(youden_t, 4),
    "rural_pos_rate": round(r_pos, 4),
    "urban_pos_rate": round(u_pos, 4),
    "demographic_parity_diff": round(abs(r_pos - u_pos), 4),
    "rural_tpr": round(r_tpr, 4),
    "urban_tpr": round(u_tpr, 4),
    "equal_opportunity_diff": round(abs(r_tpr - u_tpr), 4),
}


def build_publication_results_markdown(lb_df, fair_dict):
    """Generate publication-ready Markdown benchmark & fairness text."""
    lines = [
        "### Model Benchmark",
        "",
        "| Model | Mean AUC | SD AUC | Mean Brier |",
        "| :--- | :---: | :---: | :---: |",
    ]
    for r in lb_df.itertuples(index=False):
        lines.append(
            f"| {r.model} | {r.mean_auc:.4f} | "
            f"{r.std_auc:.4f} | {r.mean_brier:.4f} |"
        )
    lines.extend([
        "",
        "### Subgroup Fairness Summary",
        "",
        f"- **Demographic Parity Diff:** "
        f"{fair_dict['demographic_parity_diff']:.4f}",
        f"- **Equal Opportunity Diff:** "
        f"{fair_dict['equal_opportunity_diff']:.4f}",
    ])
    return "\n".join(lines)


results_md = build_publication_results_markdown(
    leaderboard_df, fairness_dict
)
print(results_md)


## Section 36.3.4 — Decision Curve Analysis (Vickers & Elkin, 2006)

In [ ]:
def compute_clinical_net_benefit(y_true, y_prob, thresholds):
    """Compute Vickers & Elkin (2006) clinical net benefit."""
    yt = np.asarray(y_true, dtype=int)
    yp = np.asarray(y_prob, dtype=float)
    n = len(yt)
    prev = float(np.mean(yt))
    rows = []
    for t in thresholds:
        t_f = float(t)
        if not (0.0 < t_f < 1.0):
            raise ValueError(
                f"Threshold {t_f} must satisfy 0 < t < 1"
            )
        pred = (yp >= t_f).astype(int)
        tp = int(np.sum((pred == 1) & (yt == 1)))
        fp = int(np.sum((pred == 1) & (yt == 0)))
        odds = t_f / (1.0 - t_f)
        nb_m = round((tp / n) - (fp / n) * odds, 5) + 0.0
        nb_a = round(prev - (1.0 - prev) * odds, 5) + 0.0
        rows.append({
            "threshold": round(t_f, 5),
            "nb_model": nb_m,
            "nb_all": nb_a,
            "nb_none": 0.0,
        })
    return pd.DataFrame(rows)


dca_df = compute_clinical_net_benefit(
    y_all, oof_rf, [0.10, 0.15, 0.20, 0.25, 0.30]
)
print(dca_df.to_string(index=False))


## Section 36.3.5 — Automated 300-Word Structured Journal Abstract

In [ ]:
def draft_structured_abstract(study_stats):
    """Format a 4-section structured medical abstract (<=300 words)."""
    n_tot = study_stats["n_total"]
    stunt_pct = study_stats["stunting_pct"]
    best_m = study_stats["best_model"]
    best_auc = study_stats["best_auc"]
    brier = study_stats["brier_score"]
    eo_gap = study_stats["eo_gap"]
    text = (
        f"**Background:** Childhood stunting remains a major maternal "
        f"and child health burden in low-resource settings.\n\n"
        f"**Methods:** We analyzed {n_tot} survey households "
        f"(stunting prevalence {stunt_pct:.1f}%) using 5-fold "
        f"cluster-grouped cross-validation comparing statistical, "
        f"tree-ensemble, and PyTorch neural models with Decision "
        f"Curve Analysis and Rural-vs-Urban fairness auditing.\n\n"
        f"**Results:** {best_m} achieved the highest discrimination "
        f"(mean ROC-AUC {best_auc:.4f}; Brier score {brier:.4f}) "
        f"with positive clinical net benefit across 10%-25% "
        f"thresholds, though an equal-opportunity gap of "
        f"{eo_gap:.4f} persisted between rural and urban strata.\n\n"
        f"**Conclusions:** Transparent, cluster-validated models "
        f"add actionable screening value when paired with subgroup "
        f"fairness monitoring and reproducible artifact releases."
    )
    wc = len(text.split())
    return {
        "abstract_text": text,
        "word_count": wc,
        "within_limit": bool(wc <= 300),
    }


top_row = leaderboard_df.iloc[0]
study_stats = {
    "n_total": int(len(df_clean)),
    "stunting_pct": round(100.0 * float(y_all.mean()), 1),
    "best_model": str(top_row["model"]),
    "best_auc": float(top_row["mean_auc"]),
    "brier_score": float(top_row["mean_brier"]),
    "eo_gap": float(fairness_dict["equal_opportunity_diff"]),
}
abstract_out = draft_structured_abstract(study_stats)
print(abstract_out["abstract_text"])
print(f"\nWords: {abstract_out['word_count']} | "
      f"Within 300-word limit?: {abstract_out['within_limit']}")


## Section 36.3.6 — Reproducible SHA-256 Artifact Bundle (Inside `TemporaryDirectory`)

In [ ]:
REQUIRED_FILES = sorted([
    "MODEL_CARD.md", "fairness_audit.json", "leaderboard.csv",
    "manifest.json", "model.joblib", "table1.csv",
])


def audit_manuscript_reproducibility(release_dir):
    """Audit release directory for all 6 required paper artifacts."""
    root = Path(release_dir)
    present = sorted(f for f in REQUIRED_FILES if (root / f).is_file())
    missing = sorted(
        f for f in REQUIRED_FILES if not (root / f).is_file()
    )
    return {
        "present": present,
        "missing": missing,
        "ready_for_submission": len(missing) == 0,
    }


with tempfile.TemporaryDirectory() as tmp_dir:
    rel = Path(tmp_dir) / "paper_release_v1"
    rel.mkdir(parents=True, exist_ok=True)
    table1_df.to_csv(rel / "table1.csv", index=False)
    leaderboard_df.to_csv(rel / "leaderboard.csv", index=False)
    (rel / "fairness_audit.json").write_text(
        json.dumps(fairness_dict, indent=2), encoding="utf-8"
    )
    final_pipe = Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("clf", RandomForestClassifier(
            n_estimators=80, max_depth=4, random_state=42
        )),
    ])
    final_pipe.fit(X_all, y_all)
    model_path = rel / "model.joblib"
    joblib.dump(final_pipe, model_path)
    sha = hashlib.sha256(model_path.read_bytes()).hexdigest()
    (rel / "MODEL_CARD.md").write_text(
        f"# Stunting Risk Model Card\n\n{results_md}\n",
        encoding="utf-8",
    )
    (rel / "config.json").write_text(
        json.dumps({"seed": 42, "features": FEATURES}, indent=2),
        encoding="utf-8",
    )
    (rel / "manifest.json").write_text(
        json.dumps(
            {"model.joblib": sha, "files": REQUIRED_FILES},
            indent=2,
        ),
        encoding="utf-8",
    )
    audit = audit_manuscript_reproducibility(rel)
    reloaded = joblib.load(model_path)
    sample_p = float(reloaded.predict_proba(X_all.iloc[:1])[0, 1])
    print("Submission ready?:", audit["ready_for_submission"])
    print("Present files:", len(audit["present"]),
          "| Missing:", audit["missing"])
    print("model.joblib SHA-256 prefix:", sha[:16])
    print("Reloaded sample P(stunted):", round(sample_p, 4))


## Section 36.3.7 — Break It On Purpose! (Capstone Pipeline Error Anatomy)

The next three cells intentionally trigger classic research-pipeline failure modes (`ValueError` on invalid DCA threshold, `ValueError` on impossible `GroupKFold` splits, and `KeyError` on an incomplete abstract dictionary) so you recognize and guard against each one.

In [ ]:
# Bug #1: ValueError — DCA threshold t=1.0 divides by (1 - t) = 0
compute_clinical_net_benefit(y_all, oof_rf, [0.15, 1.0])


In [ ]:
# Bug #2: ValueError — 5-fold GroupKFold with only 2 unique clusters
tiny_groups = np.array([1, 1, 1, 2, 2, 2])
list(GroupKFold(n_splits=5).split(X_all.iloc[:6], y_all[:6], tiny_groups))


In [ ]:
# Bug #3: KeyError — missing required 'eo_gap' key in study_stats
incomplete_stats = {k: v for k, v in study_stats.items() if k != "eo_gap"}
draft_structured_abstract(incomplete_stats)


## Section 36.4 — Stage 4 Blueprint: Complete Reproducible "First Paper" Pipeline

In [ ]:
def run_flagship_first_paper_pipeline(mch_df, seed=42):
    """Execute full CSV-to-submission pipeline inside TemporaryDirectory."""
    df = clean_mch_survey(mch_df)
    t1 = build_publication_table1(mch_df)
    lb = run_capstone_model_benchmark(mch_df, seed=seed)
    X = df[FEATURES].astype(float)
    y = df["child_stunted"].to_numpy()
    groups = df["cluster_id"].to_numpy()

    oof = np.zeros(len(y), dtype=float)
    for tr, va in GroupKFold(n_splits=5).split(X, y, groups):
        pipe = Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("clf", RandomForestClassifier(
                n_estimators=80, max_depth=4, random_state=seed
            )),
        ])
        pipe.fit(X.iloc[tr], y[tr])
        oof[va] = pipe.predict_proba(X.iloc[va])[:, 1]

    fpr, tpr, thr = roc_curve(y, oof)
    yt = float(thr[int(np.argmax(tpr - fpr))])
    yhat = (oof >= yt).astype(int)
    r_m = df["is_urban"].to_numpy() == 0
    u_m = df["is_urban"].to_numpy() == 1
    fair = {
        "threshold": round(yt, 4),
        "demographic_parity_diff": round(
            abs(float(yhat[r_m].mean() - yhat[u_m].mean())), 4
        ),
        "equal_opportunity_diff": round(
            abs(
                float(yhat[r_m & (y == 1)].mean())
                - float(yhat[u_m & (y == 1)].mean())
            ),
            4,
        ),
    }
    dca = compute_clinical_net_benefit(y, oof, [0.10, 0.15, 0.20])
    top = lb.iloc[0]
    ab = draft_structured_abstract({
        "n_total": int(len(df)),
        "stunting_pct": round(100.0 * float(y.mean()), 1),
        "best_model": str(top["model"]),
        "best_auc": float(top["mean_auc"]),
        "brier_score": float(top["mean_brier"]),
        "eo_gap": fair["equal_opportunity_diff"],
    })

    with tempfile.TemporaryDirectory() as tmp:
        rel = Path(tmp) / "release"
        rel.mkdir()
        t1.to_csv(rel / "table1.csv", index=False)
        lb.to_csv(rel / "leaderboard.csv", index=False)
        (rel / "fairness_audit.json").write_text(
            json.dumps(fair, indent=2), encoding="utf-8"
        )
        final_p = Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("scl", StandardScaler()),
            ("clf", LogisticRegression(max_iter=1000, random_state=seed)),
        ]).fit(X, y)
        joblib.dump(final_p, rel / "model.joblib")
        sha = hashlib.sha256(
            (rel / "model.joblib").read_bytes()
        ).hexdigest()
        (rel / "MODEL_CARD.md").write_text(
            ab["abstract_text"], encoding="utf-8"
        )
        (rel / "config.json").write_text(
            json.dumps({"seed": seed}, indent=2), encoding="utf-8"
        )
        (rel / "manifest.json").write_text(
            json.dumps({"model.joblib": sha}, indent=2), encoding="utf-8"
        )
        audit = audit_manuscript_reproducibility(rel)

    return {
        "n_total": int(len(df)),
        "winner": str(top["model"]),
        "winner_auc": float(top["mean_auc"]),
        "eo_gap": fair["equal_opportunity_diff"],
        "nb_at_15pct": float(dca.loc[1, "nb_model"]),
        "abstract_words": ab["word_count"],
        "ready": audit["ready_for_submission"],
        "sha256_prefix": sha[:12],
    }


print(run_flagship_first_paper_pipeline(mch_raw, seed=42))
